[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/toma-decisiones-mcda/blob/main/04_electre_iot_palmor.ipynb)

# ELECTRE, caso IoT/WSN Palmor

Elección de tecnología de comunicación (**LoRaWAN, GSM/GPRS, Sigfox, Zigbee**) para una red de sensores IoT/WSN de monitoreo agroclimático en Palmor, corregimiento de Ciénaga (Sierra Nevada de Santa Marta, Magdalena), zona con conectividad limitada verificada vía MinTIC. Los 4 criterios (Alcance de comunicación, Autonomía de batería, Infraestructura/cobertura comercial en Colombia, Madurez/viabilidad comercial del proveedor) salieron del panel de evidencia de la Sesión 1. Los valores técnicos de la matriz de decisión son reales, verificados vía WebSearch (datasheets SIMCom/DigiKey, The Things Network, Lauridsen et al. 2019 *Sensors*/MDPI, noticias de apagado de 2G en Colombia). Mismos datos y pesos que TOPSIS/VIKOR.

**Nota de alcance:** a diferencia de AHP/TOPSIS/VIKOR (ya migrados a este caso en el curso real), esta aplicación de ELECTRE al caso IoT/Palmor es una extensión nueva de este repositorio, aún no vista en clase (en el curso, S4 sigue resolviendo el caso cacao con ELECTRE). Los datos de entrada son los mismos reales de la matriz de decisión de arriba; el resultado es real (calculado con la misma fórmula e umbrales de la Sesión 4), simplemente no ha pasado por el aula todavía.

Implementación manual (no `pyDecision.algorithm.electre_i`) por la misma razón documentada en el caso cacao: su discordancia usa una normalización distinta a la fórmula enseñada en clase.

In [1]:
import itertools

tecnologias = ["LoRaWAN", "GSM/GPRS", "Sigfox", "Zigbee"]
datos = {
    "LoRaWAN":  [10,   8,   2, 5],
    "GSM/GPRS": [10.5, 0.5, 3, 2],
    "Sigfox":   [40,   2,   5, 2],
    "Zigbee":   [0.07, 1.5, 2, 4],
}
pesos = [0.16, 0.25, 0.488, 0.102]
es_beneficio = [True, True, True, True]  # las 4 son de beneficio

## Concordancia y discordancia

In [2]:
rangos = [max(datos[t][j] for t in tecnologias) - min(datos[t][j] for t in tecnologias)
          for j in range(4)]

def mejor_o_igual(a, b, j):
    return a >= b if es_beneficio[j] else a <= b

def estrictamente_mejor(a, b, j):
    return a > b if es_beneficio[j] else a < b

concordancia, discordancia = {}, {}
for a, b in itertools.permutations(tecnologias, 2):
    c = sum(w for j, w in enumerate(pesos)
            if mejor_o_igual(datos[a][j], datos[b][j], j))
    d = max([abs(datos[b][j] - datos[a][j]) / rangos[j]
             for j in range(4)
             if estrictamente_mejor(datos[b][j], datos[a][j], j)], default=0.0)
    concordancia[(a, b)] = c
    discordancia[(a, b)] = d

print("Concordancia:", {k: round(v, 2) for k, v in concordancia.items()})
print("Discordancia:", {k: round(v, 2) for k, v in discordancia.items()})

Concordancia: {('LoRaWAN', 'GSM/GPRS'): 0.35, ('LoRaWAN', 'Sigfox'): 0.35, ('LoRaWAN', 'Zigbee'): 1.0, ('GSM/GPRS', 'LoRaWAN'): 0.65, ('GSM/GPRS', 'Sigfox'): 0.1, ('GSM/GPRS', 'Zigbee'): 0.65, ('Sigfox', 'LoRaWAN'): 0.65, ('Sigfox', 'GSM/GPRS'): 1.0, ('Sigfox', 'Zigbee'): 0.9, ('Zigbee', 'LoRaWAN'): 0.49, ('Zigbee', 'GSM/GPRS'): 0.35, ('Zigbee', 'Sigfox'): 0.1}
Discordancia: {('LoRaWAN', 'GSM/GPRS'): 0.33, ('LoRaWAN', 'Sigfox'): 1.0, ('LoRaWAN', 'Zigbee'): 0.0, ('GSM/GPRS', 'LoRaWAN'): 1.0, ('GSM/GPRS', 'Sigfox'): 0.74, ('GSM/GPRS', 'Zigbee'): 0.67, ('Sigfox', 'LoRaWAN'): 1.0, ('Sigfox', 'GSM/GPRS'): 0.0, ('Sigfox', 'Zigbee'): 0.67, ('Zigbee', 'LoRaWAN'): 0.87, ('Zigbee', 'GSM/GPRS'): 0.33, ('Zigbee', 'Sigfox'): 1.0}


## Relación de superación (c* = 0.65, d* = 0.30, convención del curso)

In [3]:
c_estrella, d_estrella = 0.65, 0.30
print("a supera a b:")
for a, b in itertools.permutations(tecnologias, 2):
    if concordancia[(a, b)] >= c_estrella and discordancia[(a, b)] <= d_estrella:
        print(f"  {a} supera a {b}  (c={concordancia[(a,b)]:.2f}, d={discordancia[(a,b)]:.2f})")

a supera a b:
  LoRaWAN supera a Zigbee  (c=1.00, d=0.00)
  Sigfox supera a GSM/GPRS  (c=1.00, d=0.00)


**Resultado:** LoRaWAN supera a Zigbee, Sigfox supera a GSM/GPRS. **Sigfox y LoRaWAN quedan incomparables entre sí** (Sigfox no supera a LoRaWAN: su concordancia sí llega a 0.65, pero su discordancia es 1.00, muy por encima del umbral 0.30 — LoRaWAN gana por mucho en Autonomía, el criterio que Sigfox pierde peor). ELECTRE no entrega un ranking completo, el mismo punto pedagógico que el caso cacao ilustra: incomparabilidad real, no una falla del método.